In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT=Path.cwd().parents[1]
sys.path.insert(0,str(ROOT))

import common as cfg
import helper_funcs as hf

CONFIG=cfg.load_config()
INPUT,OUTPUT,SRC=cfg.task_paths()
OUTPUT.mkdir(parents=True,exist_ok=True)
rng=cfg.initialize(CONFIG['seed'])


In [ ]:
check=pd.read_csv(INPUT/'j_posterior_predict'/'posterior_predictive_check.csv')
ranking=pd.read_csv(INPUT/'j_posterior_predict'/'promising_hypotheses.csv')
particles=pd.read_csv(INPUT/'h_abc_smc'/'abc_particles.csv')
targets=pd.read_csv(INPUT/'b_prepare'/'empirical_targets.csv')
followup=pd.read_csv(INPUT/'b_prepare'/'followup_distribution.csv')
best=ranking.iloc[0]['model']


In [ ]:
loo=[]
blocks=sorted(check['block'].unique())
for excluded in ['none',*blocks]:
    subset=check if excluded=='none' else check.loc[check['block']!=excluded]
    score=subset.groupby('model',as_index=False)['median_standardized_error'].median().rename(columns={'median_standardized_error':'error'})
    score['rank']=score['error'].rank(method='min').astype(int)
    score['excluded_block']=excluded
    loo.append(score)
leave_one_out=pd.concat(loo,ignore_index=True)


In [ ]:
final=particles.loc[(particles['model']==best)&(particles['generation']==particles.loc[particles['model']==best,'generation'].max())]
sampled=hf.weighted_sample(final,CONFIG['sensitivity_draws'],rng)
stability=[]
for n in [max(128,CONFIG['n_scholars']//2),CONFIG['n_scholars'],min(8192,CONFIG['n_scholars']*2)]:
    for draw,row in sampled.iterrows():
        theta={name:row[name] for name in hf.prior_specs(best)}
        _,distance,blocks_distance,_=hf.simulate_summary(best,theta,CONFIG,followup,targets,rng,n=n)
        stability.append({'model':best,'n_scholars':n,'draw':draw,'distance':distance})
stability=pd.DataFrame(stability)


In [ ]:
width=[]
for model in CONFIG['models']:
    for scale in [0.75,1.0,1.25]:
        model_rng=np.random.default_rng(cfg.model_seed(CONFIG['seed'],model,int(scale*100)))
        draws,_,_=hf.prior_predictive(model,CONFIG,followup,targets,model_rng,draws=CONFIG['sensitivity_draws'],prior_scale=scale)
        width.append({'model':model,'prior_z_scale':scale,'median_distance':draws['distance'].median(),'best_distance':draws['distance'].min()})
prior_width=pd.DataFrame(width)


In [ ]:
leave_one_out.to_csv(OUTPUT/'leave_one_block_out.csv',index=False)
stability.to_csv(OUTPUT/'simulation_size_stability.csv',index=False)
prior_width.to_csv(OUTPUT/'prior_width_screening.csv',index=False)


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
sns.lineplot(data=leave_one_out,x='excluded_block',y='rank',hue='model',style='model',markers=True,ax=axes[0])
sns.boxplot(data=stability,x='n_scholars',y='distance',ax=axes[1])
axes[0].tick_params(axis='x',rotation=60)
axes[0].invert_yaxis()
axes[0].set(xlabel='target block omitted',ylabel='descriptive rank')
axes[1].set(xlabel='simulated scholars',ylabel='posterior-predictive distance')
fig.tight_layout()
fig.savefig(OUTPUT/'sensitivity.png',dpi=220)
plt.close(fig)
leave_one_out.loc[leave_one_out['rank']==1]
